In [83]:
import pandas as pd
import numpy as np
import glob
import time
import random
import os
import musicbrainzngs
from SPARQLWrapper import SPARQLWrapper, JSON
import urllib.parse
import urllib.request
import json
import requests
from requests.utils import quote

In [84]:
wb = pd.read_csv("../../data/raw/world-bank-data/raw-world-bank.csv", header=None)

num_years = wb.shape[1] - 4
year_cols = [f"year_{i}" for i in range(num_years)]
wb.columns = ["indicator_name", "series_code", "country", "country_code"] + year_cols

wb.replace("..", np.nan, inplace=True)

wb_long = wb.melt(
    id_vars=["indicator_name", "series_code", "country", "country_code"],
    value_vars=year_cols,
    var_name="year_idx",
    value_name="value"
)

wb_long["value"] = pd.to_numeric(wb_long["value"], errors="coerce")

start_year = 2019
wb_long["year"] = wb_long["year_idx"].str.extract("(\d+)").astype(int) + start_year

wb_long = wb_long.drop(columns=["year_idx"])

wb_clean = wb_long.pivot_table(
    index=["country", "country_code", "year"],
    columns="series_code",
    values="value"
).reset_index()

wb_clean.columns.name = None

wb_clean.to_csv("../../data/interim/interim-world-bank.csv", index=False)

print("Data cleaned, missing values handled, and saved to '../../data/interim/interim-world-bank.csv'")

Data cleaned, missing values handled, and saved to '../../data/interim/interim-world-bank.csv'


The code above takes the raw data I gathered from World Bank for several economic/internet usage indicators for each of the countries from 2019-2024 and cleans + widens it into a format better suited for data analysis. This data is then saved to a new intermediate file for later aggregation with the Spotify datasets once they are done.

In [85]:
musicbrainzngs.set_useragent("ArtistCountryFetcher", "1.0", "cielo@uchicago.edu")

# wikidata SPARQL endpoint
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")

# load all unique artists from spotify files
artist_set = set()

for year in range(2019, 2026):
    filename = f"../../data/interim/merged_data_{year}.csv"
    df = pd.read_csv(filename)
    
    for names in df['artist_names'].dropna():
        for artist in names.split(","):
            artist_set.add(artist.strip())
print(f"Found {len(artist_set)} unique artists across all files.")

output_file = "../../data/interim/artist_countries.csv"

# load existing results if alr exists
if os.path.exists(output_file):
    existing_df = pd.read_csv(output_file)
    existing_artists = set(existing_df['artist_name'])
    file_exists = True
else:
    existing_artists = set()
    file_exists = False

artists_to_query = artist_set - existing_artists
print(f"{len(artists_to_query)} artists left to query.")

# query apis and append to file
for artist in artists_to_query:
    country = None

    # musicbrainz lookup
    try:
        res = musicbrainzngs.search_artists(artist=artist, limit=1)
        if res['artist-list']:
            country = res['artist-list'][0].get('country')
        time.sleep(random.uniform(1, 3))
    except Exception as e:
        print(f"MB error for {artist}: {e}")

    # wikidata fallback
    if not country:
        try:
            query = f"""
            SELECT ?countryLabel WHERE {{
              ?artist ?p wd:Q5.
              ?artist rdfs:label "{artist}"@en.
              ?artist wdt:P27 ?country.
              SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
            }}
            """
            sparql.setQuery(query)
            sparql.setReturnFormat(JSON)
            response = sparql.query().convert()
            bindings = response["results"]["bindings"]
            if bindings:
                country = bindings[0]["countryLabel"]["value"]
            time.sleep(random.uniform(1, 2))
        except Exception as e:
            print(f"Wikidata error for {artist}: {e}")

    df_partial = pd.DataFrame([{"artist_name": artist, "country": country}])

    df_partial.to_csv(
        output_file,
        mode='a' if file_exists else 'w',
        header=not file_exists,
        index=False
    )

    file_exists = True

print("Done! Results saved in", output_file)

Found 5752 unique artists across all files.
0 artists left to query.
Done! Results saved in ../../data/interim/artist_countries.csv


The code above extracts all unique artists from each of the yearly Spotify data files and queries their country of origin from the MusicBrainz artist API endpoint or falls back upon a Wikidata query if an error occurs during the former process. Unique artists and their country of origin are saved in the `artist_countries.csv` file in the `data/interim/` directory for later mapping to the merged Spotify/World Bank dataset. The code above takes quite a bit of time to run since it queries for thousands of artists through 2 endpoints.

NOTE: tracks in the Spotify files may have more than one artist; the code above ensures that artist values are split in these cases to ensure unique artists.

In [86]:
CSV_PATH = "../../data/interim/artist_countries.csv"
# reusable safe http request
def safe_request(url, headers=None, max_retries=5):
    headers = headers or {'User-Agent': 'Mozilla/5.0'}
    
    for attempt in range(1, max_retries + 1):
        try:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=10) as r:
                return r.read().decode()
        except Exception as e:
            print(f"Request error (attempt {attempt}): {e}")
            sleep_time = 1.5 * attempt + random.random()
            print(f"Sleeping {sleep_time:.2f}s before retry...")
            time.sleep(sleep_time)

    return None


# musicbrainz query
def query_musicbrainz(artist):
    print("MB lookup...")

    headers = {
        "User-Agent": "YourAppName/1.0 ( your-email@example.com )"
    }

    # Encode artist properly
    artist_query = f'artist:"{artist}"'

    params = {
        "query": artist_query,
        "fmt": "json"
    }

    try:
        r = requests.get(
            "https://musicbrainz.org/ws/2/artist/",
            params=params,
            headers=headers,
            timeout=10
        )

        if r.status_code != 200:
            print(f"MB returned HTTP {r.status_code}")
            return None

        data = r.json()

        if data.get("artists"):
            return data["artists"][0].get("country")
        return None

    except Exception as e:
        print(f"MB EXCEPTION: {e}")
        return None



# wikidata query
def query_wikidata(artist):
    print("WD lookup...")
    
    query = f"""
    SELECT ?countryLabel WHERE {{
        ?artist rdfs:label "{artist}"@en.
        OPTIONAL {{ ?artist wdt:P27 ?country. }}
        SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    LIMIT 1
    """
    
    url = (
        "https://query.wikidata.org/sparql"
        "?format=json&query=" + urllib.parse.quote(query)
    )

    headers = {'User-Agent': 'Mozilla/5.0'}

    data_raw = safe_request(url, headers=headers)
    if not data_raw:
        print("Wikidata failed (no response)")
        return None

    try:
        results = json.loads(data_raw)
        bindings = results["results"]["bindings"]
        if bindings:
            return bindings[0]["countryLabel"]["value"]
    except Exception as e:
        print(f"WD EXCEPTION: {e}")

    return None


def update_missing_artists():
    print("LOADING existing CSV…")

    df = pd.read_csv(CSV_PATH)

    missing_mask = df['country'].isna() | (df['country'] == "")
    artists_missing = df[missing_mask]['artist_name'].tolist()

    print(f"Found {len(artists_missing)} artists missing countries.\n")

    for artist in artists_missing:
        # Try MusicBrainz first
        country = query_musicbrainz(artist)
        time.sleep(1.2)

        # fallback: Wikidata
        if not country:
            country = query_wikidata(artist)
            time.sleep(1.2)

        # Still nothing → UNKNOWN
        if not country:
            country = "UNKNOWN"

        # Save immediately
        df.loc[df['artist_name'] == artist, 'country'] = country
        df.to_csv(CSV_PATH, index=False)

        print(f"Saved: {artist} → {country}")

    print("\nDONE! CSV fully updated, restart-safe.")


update_missing_artists()

LOADING existing CSV…
Found 0 artists missing countries.


DONE! CSV fully updated, restart-safe.


The code above goes through artists that didn't get countries saved due to API request overload and tries to extract data using retry-safe methods. If the country of origin is still not able to be obtained due to the information simply not being available within the APIs, it is simply saved as 'UNKNOWN.'

In [87]:
artist_countries = pd.read_csv(CSV_PATH)
len(artist_countries.loc[artist_countries.country == 'UNKNOWN'])/len(artist_countries)

0.19071627260083449

Approximately 19% of unique artists are missing countries of origin. This will, of course, impact how I analyze domestic vs. global music consumption trends, but at least there is a decent amount of data to work with.

In [88]:
spotify_file = '../../data/interim/spotify_global_merged.csv'
artist_country_file = '../../data/interim/artist_countries.csv'
output_file = '../../data/interim/spotify_with_artist_countries.csv'

spotify_df = pd.read_csv(spotify_file)
artist_df = pd.read_csv(artist_country_file)

artist_country_map = dict(zip(artist_df['artist_name'], artist_df['country']))

spotify_df = spotify_df.drop(columns=['uri'])

def domestic_foreign_flags(artist_names, track_country_code, track_country_name):
    artists = [a.strip() for a in artist_names.split(',')]
    domestic = 0
    foreign = 0
    for artist in artists:
        artist_country = artist_country_map.get(artist)
        if artist_country:
            # consider domestic if artist_country matches either code or name
            if artist_country == track_country_code or artist_country.lower() == str(track_country_name).lower():
                domestic = 1
            else:
                foreign = 1
    return pd.Series({'domestic': domestic, 'foreign': foreign})

flags_df = spotify_df.apply(
    lambda row: domestic_foreign_flags(row['artist_names'], row['country_code'], row['country_name']),
    axis=1
)

# Merge the flags back into the dataframe
spotify_df = pd.concat([spotify_df, flags_df], axis=1)

spotify_df.to_csv(output_file, index=False)

print(f"Processed file saved to {output_file}")

KeyboardInterrupt: 

In [ ]:
spotify_file = '../../data/interim/spotify_with_artist_countries.csv'
wb_file = '../../data/interim/interim-world-bank.csv'
output_file = '../../data/processed/spotify_merged_with_wb.csv'

spotify_df = pd.read_csv(spotify_file)
wb_df = pd.read_csv(wb_file)

spotify_df['year'] = pd.to_numeric(spotify_df['year'], errors='coerce')
wb_df['year'] = pd.to_numeric(wb_df['year'], errors='coerce')

merged_df = spotify_df.merge(
    wb_df,
    left_on=['country_name', 'year'],
    right_on=['country', 'year'],
    how='left',
    suffixes=('', '_wb')
)

merged_df = merged_df.drop(columns=['country_wb', 'source_file'], errors='ignore')

merged_df.to_csv(output_file, index=False)
print(f"Merged dataset saved to {output_file}")
merged_df.head()

Merged dataset saved to ../../data/processed/spotify_merged_with_wb.csv


,rank,artist_names,track_name,source,peak_rank,previous_rank,weeks_on_chart,streams,country_code,country_name,...,SI.POV.GINI,SP.POP.GROW,SP.POP.TOTL,SP.RUR.TOTL.ZG,SP.RUR.TOTL.ZS,SP.URB.GROW,SP.URB.TOTL.IN.ZS,ST.INT.ARVL,ST.INT.DPRT,ST.INT.XPND.CD
0,1,"Pedro Capó, Farruko",Calma - Remix,Sony Music Latin,1,2,10,2506857,AR,Argentina,...,43.3,0.710901,44973465.0,-0.788598,8.009,0.842522,91.991,7399000.0,15352000.0,9.845000e+09
1,2,Paulo Londra,Adan y Eva,WEA Latina,1,1,10,2451964,AR,Argentina,...,43.3,0.710901,44973465.0,-0.788598,8.009,0.842522,91.991,7399000.0,15352000.0,9.845000e+09
2,3,DJ Alex,Leña para el Carbón,DJ Alex,3,3,5,1709216,AR,Argentina,...,43.3,0.710901,44973465.0,-0.788598,8.009,0.842522,91.991,7399000.0,15352000.0,9.845000e+09
3,4,"Bad Bunny, Drake",MIA,Rimas Entertainment LLC,2,5,13,1427528,AR,Argentina,...,43.3,0.710901,44973465.0,-0.788598,8.009,0.842522,91.991,7399000.0,15352000.0,9.845000e+09
4,5,"Duki, DrefQuila",Sin Culpa (feat.DrefQuila),SSJ Records / DALE PLAY Records,5,6,5,1389999,AR,Argentina,...,43.3,0.710901,44973465.0,-0.788598,8.009,0.842522,91.991,7399000.0,15352000.0,9.845000e+09


In [ ]:
merged_df = merged_df[merged_df['year'] != 2025]
len(merged_df)

23800

I have dropped all data from 2025 because the World Bank data was not available for 2025 yet.

In [ ]:
missing_summary = (
    merged_df.isna()
      .agg(['sum', 'mean'])
      .T
      .rename(columns={'sum': 'missing_count', 'mean': 'missing_percent'})
)

missing_summary['missing_percent'] = (missing_summary['missing_percent'] * 100).round(2)

missing_summary.sort_values('missing_count', ascending=False)

,missing_count,missing_percent
ST.INT.XPND.CD,18850.0,79.20
ST.INT.DPRT,18450.0,77.52
ST.INT.ARVL,17650.0,74.16
SE.ADT.LITR.ZS,17300.0,72.69
SE.ADT.1524.LT.ZS,17300.0,72.69
GC.TAX.INTT.RV.ZS,14050.0,59.03
SI.POV.GINI,11900.0,50.00
IT.CEL.SETS.P2,5750.0,24.16
IT.NET.BBND.P2,5750.0,24.16
IT.NET.USER.ZS,4350.0,18.28


As we can see, there is a large amoung of missing data for the following (75-80%):
- ST.INT.XPND.CD (International tourism, expenditures (current US$))
- ST.INT.DPRT (International tourism, departures)
- ST.INT.ARVL (International tourism, arrivals)
- SE.ADT.LITR.ZS (Literacy rate, adult total (% of people ages 15 and above))
- SE.ADT.1524.LT.ZS (Literacy rate, youth total (% of people ages 15-24))

This is because these indicators do not have data for multiple countries in certain years. I will be dropping these columns for analysis since they have such high missingness (could be unreliable and lead to biased results).

The next indicators with the second largest amount of missing data (50-60%) are:
- GC.TAX.INTT.RV.ZS (Taxes on international trade (% of revenue))
- SI.POV.GINI (Gini index)

I'm going to try to impute these using median values per country or year. Country medians reserve cross-country differences while year medians preserve temporal trends. I'd really like to avoid dropping these since they could be great economic indicators.

Finally, the indicators with the least amount of missing data (8-24%) are:
- IT.CEL.SETS.P2, IT.NET.BBND.P2, IT.NET.USER.ZS (Mobile cellular subscriptions per 100 people, Fixed broadband subscriptions per 100 people, Individuals using the Internet (% of population))
- NE.IMP.GNFS.ZS, NE.EXP.GNFS.ZS (Imports of goods and services (% of GDP), Exports of goods and services (% of GDP))
- And many other growth indicators (Population growth, Rural population total, Urban population growth, GDP growth (annual %), Foreign direct investment--net inflows (% of GDP), Foreign direct investment--net outflows (% of GDP), etc.)

These should be the easiest to work with. I'll attempt to do simple imputations such as median imputations per year (to preserve trends over time) or median/mean per country (to preserve country-specific differences).

The core of my Spotify data has no missingness since I took it directly from the Spotify Top 50 Songs API.

In [ ]:
# dropping indicators with highest missing data
cols_to_drop = ['ST.INT.XPND.CD','ST.INT.DPRT','ST.INT.ARVL','SE.ADT.LITR.ZS','SE.ADT.1524.LT.ZS']
merged_df.drop(columns=cols_to_drop, inplace=True)

# attempting to impute for gini index and taxes on international trade by country median
for col in ['GC.TAX.INTT.RV.ZS','SI.POV.GINI']:
    merged_df[col] = merged_df.groupby('country')[col].transform(lambda x: x.fillna(x.median()))

# imputing data for columns with least amount of missing data (within-year)
low_missing_cols = ['IT.CEL.SETS.P2','IT.NET.BBND.P2','IT.NET.USER.ZS','SP.RUR.TOTL.ZG',
                    'NE.IMP.GNFS.ZS','NE.EXP.GNFS.ZS','SP.POP.GROW','SP.POP.TOTL',
                    'NY.GDP.DEFL.KD.ZG','NY.GDP.DEFL.KD.ZG.AD','NY.GDP.MKTP.KD.ZG',
                    'NY.GDP.PCAP.CD','SP.URB.GROW','BX.KLT.DINV.WD.GD.ZS','BM.KLT.DINV.WD.GD.ZS',
                    'SP.URB.TOTL.IN.ZS','SP.RUR.TOTL.ZS','IT.NET.SECR']
for col in low_missing_cols:
    merged_df[col] = merged_df.groupby('year')[col].transform(lambda x: x.fillna(x.median()))

merged_df.head(5)

C:\Users\HVNLY\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\HVNLY\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\HVNLY\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\HVNLY\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarnin

,rank,artist_names,track_name,source,peak_rank,previous_rank,weeks_on_chart,streams,country_code,country_name,...,NY.GDP.DEFL.KD.ZG.AD,NY.GDP.MKTP.KD.ZG,NY.GDP.PCAP.CD,SI.POV.GINI,SP.POP.GROW,SP.POP.TOTL,SP.RUR.TOTL.ZG,SP.RUR.TOTL.ZS,SP.URB.GROW,SP.URB.TOTL.IN.ZS
0,1,"Pedro Capó, Farruko",Calma - Remix,Sony Music Latin,1,2,10,2506857,AR,Argentina,...,49.195579,-2.000861,9955.974787,43.3,0.710901,44973465.0,-0.788598,8.009,0.842522,91.991
1,2,Paulo Londra,Adan y Eva,WEA Latina,1,1,10,2451964,AR,Argentina,...,49.195579,-2.000861,9955.974787,43.3,0.710901,44973465.0,-0.788598,8.009,0.842522,91.991
2,3,DJ Alex,Leña para el Carbón,DJ Alex,3,3,5,1709216,AR,Argentina,...,49.195579,-2.000861,9955.974787,43.3,0.710901,44973465.0,-0.788598,8.009,0.842522,91.991
3,4,"Bad Bunny, Drake",MIA,Rimas Entertainment LLC,2,5,13,1427528,AR,Argentina,...,49.195579,-2.000861,9955.974787,43.3,0.710901,44973465.0,-0.788598,8.009,0.842522,91.991
4,5,"Duki, DrefQuila",Sin Culpa (feat.DrefQuila),SSJ Records / DALE PLAY Records,5,6,5,1389999,AR,Argentina,...,49.195579,-2.000861,9955.974787,43.3,0.710901,44973465.0,-0.788598,8.009,0.842522,91.991


In [ ]:
missing_summary_after = merged_df.isna().mean().sort_values(ascending=False) * 100
missing_summary_after.head(10)

GC.TAX.INTT.RV.ZS    49.789916
SI.POV.GINI          22.268908
IT.CEL.SETS.P2       17.647059
IT.NET.BBND.P2       17.647059
country               8.193277
country_code_wb       8.193277
artist_names          0.000000
rank                  0.000000
source                0.000000
track_name            0.000000
dtype: float64

It seems approximately half of `GC.TAX.INTT.RV.ZS` is still missing. I'll try to impute by country median to preserve cross-country tax structure differences (I'd like to avoid dropping this column since it could be a useful globalization measure).

I'll similarly be imputing country medians for the Gini index indicator since it now has relatively manageable missingness.

I'll impute year medians for both `IT.CEL.SETS.P2` and 'IT.NET.BBND.P2` since they also have relatively manageable missingness. It might reflect the global adoption trend of mobile phones (especially since all these countries have access to Spotify).

The `country` and `country_code_wb` columns can be easily fixed by mapping between countries and their alpha codes.

In [ ]:
COUNTRY_MAP = {
    "US": "United States", "CA": "Canada", "MX": "Mexico",
    "BR": "Brazil", "AR": "Argentina", "CO": "Colombia",
    "GB": "United Kingdom", "DE": "Germany", "FR": "France",
    "SE": "Sweden", "ES": "Spain", "IT": "Italy",
    "JP": "Japan", "KR": "South Korea", "IN": "India",
    "SG": "Singapore", "ZA": "South Africa", "NG": "Nigeria",
    "AU": "Australia", "TR": "Turkey", "PL": "Poland"
}

merged_df["country"] = merged_df["country_code"].map(COUNTRY_MAP)
merged_df["country_code_wb"] = merged_df["country_code"]

# group by country, then fallback to overall median
# tax revenue
merged_df['GC.TAX.INTT.RV.ZS'] = merged_df.groupby('country')['GC.TAX.INTT.RV.ZS'].transform(
    lambda x: x.fillna(x.median())
)
merged_df['GC.TAX.INTT.RV.ZS'] = merged_df['GC.TAX.INTT.RV.ZS'].fillna(merged_df['GC.TAX.INTT.RV.ZS'].median())

# gini index
merged_df['SI.POV.GINI'] = merged_df.groupby('country')['SI.POV.GINI'].transform(
    lambda x: x.fillna(x.median())
)
merged_df['SI.POV.GINI'] = merged_df['SI.POV.GINI'].fillna(merged_df['SI.POV.GINI'].median())

# cellular subscriptions
merged_df['IT.CEL.SETS.P2'] = merged_df.groupby('year')['IT.CEL.SETS.P2'].transform(
    lambda x: x.fillna(x.median())
)
merged_df['IT.CEL.SETS.P2'] = merged_df['IT.CEL.SETS.P2'].fillna(merged_df['IT.CEL.SETS.P2'].median())

# Broadband subscriptions: group by year, fallback to overall median
merged_df['IT.NET.BBND.P2'] = merged_df.groupby('year')['IT.NET.BBND.P2'].transform(
    lambda x: x.fillna(x.median())
)
merged_df['IT.NET.BBND.P2'] = merged_df['IT.NET.BBND.P2'].fillna(merged_df['IT.NET.BBND.P2'].median())

merged_df.isna().mean().sort_values(ascending=False) * 100

C:\Users\HVNLY\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\HVNLY\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\HVNLY\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\HVNLY\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarnin

rank                    0.0
artist_names            0.0
track_name              0.0
source                  0.0
peak_rank               0.0
previous_rank           0.0
weeks_on_chart          0.0
streams                 0.0
country_code            0.0
country_name            0.0
quarter                 0.0
year                    0.0
domestic                0.0
foreign                 0.0
country                 0.0
country_code_wb         0.0
BM.KLT.DINV.WD.GD.ZS    0.0
BX.KLT.DINV.WD.GD.ZS    0.0
GC.TAX.INTT.RV.ZS       0.0
IT.CEL.SETS.P2          0.0
IT.NET.BBND.P2          0.0
IT.NET.SECR             0.0
IT.NET.USER.ZS          0.0
NE.EXP.GNFS.ZS          0.0
NE.IMP.GNFS.ZS          0.0
NY.GDP.DEFL.KD.ZG       0.0
NY.GDP.DEFL.KD.ZG.AD    0.0
NY.GDP.MKTP.KD.ZG       0.0
NY.GDP.PCAP.CD          0.0
SI.POV.GINI             0.0
SP.POP.GROW             0.0
SP.POP.TOTL             0.0
SP.RUR.TOTL.ZG          0.0
SP.RUR.TOTL.ZS          0.0
SP.URB.GROW             0.0
SP.URB.TOTL.IN.ZS   

For `GC.TAX.INTT.RV.WD.GD.ZS` (corporate tax revenue) and `SI.POV.GINI` (income inequality), I applied country-level median imputation, which preserves typical values within each country while accounting for country-specific economic characteristics. For `IT.CEL.SETS.P2` (cellular subscriptions) and `IT.NET.BBND.P2` (broadband subscriptions), I used year-level median imputation, reflecting the general temporal trends in technology adoption across countries. In all cases, any remaining missing values were filled with the overall median, providing a practical fallback to maintain a complete dataset for analysis.

This is, of course, NOT a perfect solution and will not be entirely accurate for countries that had missing data, but it at least balance some realism (country and year-specific patterns) with practicality (ensuring no missing values disrupt downstream visualization or analysis, especially since I need as much data as possible for these goals).

Yay! The data has now been cleaned and is ready for use.

In [ ]:
merged_df.to_csv('../../data/processed/spotify-wb-cleaned.csv', index=False)